# Business Entity Resolution — Train + Infer Pipeline

Assumes `preprocessing.py`, `features.py`, `blocking.py`, `metrics.py`, `train.py`, `predict.py` are in the same directory (or on `sys.path`) as this notebook.

In [3]:
import os
print(os.getcwd())
print(os.listdir("."))

C:\Projects\Amazon\student_resource\notebook
['.ipynb_checkpoints', 'entity_resolution_pipeline.ipynb', 'output']


In [4]:
import sys
import os
import json
import numpy as np
import pandas as pd

SRC_DIR = r"C:\Projects\Amazon\student_resource\code\business_entity_resolution\src"

sys.path.insert(0, SRC_DIR)

print("Python files location:")
print(SRC_DIR)
print(os.listdir(SRC_DIR))

from preprocessing import normalize_name, normalize_address
from blocking import generate_candidates
from features import build_pair_features, ALL_FEATURE_COLUMNS
from train import load_ground_truth, label_pairs, split_source1_ids, detect_cuda, build_model
from metrics import predictions_from_scored_pairs, entity_level_report, sweep_threshold

pd.set_option("display.max_colwidth", 80)

Python files location:
C:\Projects\Amazon\student_resource\code\business_entity_resolution\src
['.ipynb_checkpoints', 'blocking.py', 'features.py', 'metrics.py', 'predict.py', 'preprocessing.py', 'train.py', '__pycache__']


In [5]:
DATA_DIR = "dataset/train"
TEST_DIR = "dataset/test"
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

DEV_MODE = True          # sample down while iterating on correctness; flip False for the real run
DEV_SAMPLE_SIZE = 5000

TOP_K = 30                # blocking: tfidf top-k candidates per S1 entity
VAL_FRAC = 0.2
RANDOM_STATE = 42

USE_CUDA = detect_cuda()
print("CUDA available:", USE_CUDA)


CUDA available: True


In [6]:
import os

BASE_DIR = r"C:\Projects\Amazon\student_resource"

DATA_DIR = os.path.join(BASE_DIR, "dataset", "train")
TEST_DIR = os.path.join(BASE_DIR, "dataset", "test")
OUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_DIR =", DATA_DIR)
print("Exists =", os.path.exists(DATA_DIR))
print("Files =", os.listdir(DATA_DIR))

DATA_DIR = C:\Projects\Amazon\student_resource\dataset\train
Exists = True
Files = ['train_ground_truth.tsv', 'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv']


In [7]:

try:
    source1 = pd.read_csv(f"{DATA_DIR}/train_source1.tsv", sep="\t", dtype=str, engine="pyarrow")
    source2 = pd.read_csv(f"{DATA_DIR}/train_source2.tsv", sep="\t", dtype=str, engine="pyarrow")
    source3 = pd.read_csv(f"{DATA_DIR}/train_source3.tsv", sep="\t", dtype=str, engine="pyarrow")
except (ImportError, ValueError):
    source1 = pd.read_csv(f"{DATA_DIR}/train_source1.tsv", sep="\t", dtype=str)
    source2 = pd.read_csv(f"{DATA_DIR}/train_source2.tsv", sep="\t", dtype=str)
    source3 = pd.read_csv(f"{DATA_DIR}/train_source3.tsv", sep="\t", dtype=str)

ground_truth = load_ground_truth(f"{DATA_DIR}/train_ground_truth.tsv")

print(f"S1={len(source1)}  S2={len(source2)}  S3={len(source3)}  GT rows={len(ground_truth)}")
source1.head()


S1=2206821  S2=5034616  S3=5285603  GT rows=2206821


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India


In [8]:
if DEV_MODE:
    source1_full = source1
    source1 = source1.sample(min(DEV_SAMPLE_SIZE, len(source1)), random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"DEV_MODE on: sampled S1 down to {len(source1)} rows (full={len(source1_full)})")


DEV_MODE on: sampled S1 down to 5000 rows (full=2206821)


## Blocking — candidate generation
`generate_candidates` runs a token-overlap pass and a TF-IDF top-k pass over the pooled S2+S3 records, then merges them. S1 entities with zero candidates get one row with `candidate_entity_id = None` so they survive as declared singletons.

In [ ]:
%%time
candidate_pairs = generate_candidates(source1, source2, source3, top_k=TOP_K)

n_with_cands = candidate_pairs.dropna(subset=["candidate_entity_id"])["source1_entity_id"].nunique()
n_total = source1["entity_id"].nunique()
print(f"S1 entities with >=1 candidate: {n_with_cands}/{n_total}")
print(f"Total candidate pairs: {len(candidate_pairs.dropna(subset=['candidate_entity_id']))}")
candidate_pairs.head()


In [ ]:
labeled_pairs = candidate_pairs.dropna(subset=["candidate_entity_id"]).copy()
labeled_pairs = label_pairs(labeled_pairs, ground_truth)

# blocking recall ceiling: of the S1 entities that HAVE ground-truth matches,
# what fraction had at least one true match survive into the candidate set?
gt_with_matches = {k for k, v in ground_truth.items() if v and k in set(source1["entity_id"])}
recovered = labeled_pairs.loc[labeled_pairs["label"] == 1, "source1_entity_id"].unique()
recall_ceiling = len(set(recovered) & gt_with_matches) / max(len(gt_with_matches), 1)
print(f"Blocking recall ceiling on this S1 sample: {recall_ceiling:.4f}")
print(f"Positive pairs: {(labeled_pairs['label']==1).sum()}  Negative pairs: {(labeled_pairs['label']==0).sum()}")


## Train / validation split
Split at the **entity level**, not the pair level — otherwise some of an S1 entity's candidates end up in train and others in val, leaking that entity's identity across the split.

In [ ]:
train_ids, val_ids = split_source1_ids(source1, val_frac=VAL_FRAC, random_state=RANDOM_STATE)

train_pairs = labeled_pairs[labeled_pairs["source1_entity_id"].isin(train_ids)].reset_index(drop=True)
val_pairs   = labeled_pairs[labeled_pairs["source1_entity_id"].isin(val_ids)].reset_index(drop=True)

print(f"Train entities: {len(train_ids)}  Val entities: {len(val_ids)}")
print(f"Train pairs: {len(train_pairs)}  Val pairs: {len(val_pairs)}")


In [ ]:
%%time
records = pd.concat([source1, source2, source3], ignore_index=True).set_index("entity_id")

train_feat = build_pair_features(train_pairs, records)
val_feat   = build_pair_features(val_pairs, records)

X_train, y_train = train_feat[ALL_FEATURE_COLUMNS], train_feat["label"]
X_val,   y_val   = val_feat[ALL_FEATURE_COLUMNS],   val_feat["label"]

print(X_train.shape, X_val.shape)


## Cell 6 — train the model
CUDA is used automatically when available (`USE_CUDA` from Cell 2); falls back to CPU `hist` otherwise.

In [ ]:
%%time
model = build_model(use_cuda=USE_CUDA)
try:
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
except Exception as e:
    # GPU build/driver mismatch -> fall back to CPU rather than losing the run
    print(f"CUDA fit failed ({e}); falling back to CPU.")
    model = build_model(use_cuda=False)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

val_df = val_feat.copy()


## Cell 6b — pair-level sanity check
This is a check on the raw pairwise classifier only — **not** the competition metric. The real score is the entity-level macro F₀.5 computed next.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

val_df["score"] = model.predict_proba(X_val)[:, 1]

print("Pair-level ROC-AUC:   ", round(roc_auc_score(y_val, val_df["score"]), 4))
print("Pair-level PR-AUC:    ", round(average_precision_score(y_val, val_df["score"]), 4))

pred_binary = (val_df["score"] >= 0.5).astype(int)
print("\nClassification report @ 0.5 (pair-level, NOT the final metric):")
print(classification_report(y_val, pred_binary, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_val, pred_binary))


## Cell 7 — threshold sweep for the real metric (macro F₀.5)
`val_gt` is built from every **val entity**, not just the ones with surviving candidates, so entities the blocking stage missed still count as (correctly or incorrectly) predicted singletons.

In [ ]:
val_gt = {eid: ground_truth.get(eid, set()) for eid in val_ids}

best_t, best_f05, sweep_results = sweep_threshold(val_df, val_gt, margin_min=0.0)
print(f"Best threshold = {best_t:.2f}   Validation macro F0.5 = {best_f05:.4f}")

import matplotlib.pyplot as plt
xs, ys = zip(*sweep_results)
plt.figure(figsize=(6, 4))
plt.plot(xs, ys)
plt.axvline(best_t, color="red", linestyle="--", label=f"best={best_t:.2f}")
plt.xlabel("threshold"); plt.ylabel("macro F0.5"); plt.legend(); plt.title("Threshold sweep")
plt.show()


In [ ]:
preds = predictions_from_scored_pairs(val_df, threshold=best_t)
report = entity_level_report(preds, val_gt)

print(f"Entities evaluated:     {report['n_entities']}")
print(f"Macro F0.5:             {report['macro_f05']:.4f}")
print(f"Macro precision:        {report['macro_precision']:.4f}")
print(f"Macro recall:           {report['macro_recall']:.4f}")
print(f"Perfect entities (1.0): {report['n_perfect']} ({report['n_perfect']/report['n_entities']:.1%})")
print(f"Zero-score entities:    {report['n_zero']} ({report['n_zero']/report['n_entities']:.1%})")
print(f"Singleton accuracy:     {report['singleton_accuracy']:.4f}  (n={report['singleton_total']})")


## Cell 7c — margin filter, does it help?
Optional: only keep a candidate if it beats the runner-up by `margin_min`. Try a couple of values; keep whichever scores higher.

In [ ]:
for m in (0.05, 0.1, 0.15, 0.2):
    t_m, f05_m, _ = sweep_threshold(val_df, val_gt, margin_min=m)
    print(f"margin_min={m:.2f}  threshold={t_m:.2f}  F0.5={f05_m:.4f}")
print(f"No margin filter:  threshold={best_t:.2f}  F0.5={best_f05:.4f}")

# set these to whichever combination scored highest above
FINAL_THRESHOLD = best_t
FINAL_MARGIN_MIN = 0.0


## Cell 7d — look at actual errors
The most useful cell for improving the model. False negatives (FN) point to gaps in blocking or weak similarity features; false positives (FP) point to a threshold or feature that doesn't discriminate near-duplicate names well.

In [ ]:
preds = predictions_from_scored_pairs(val_df, threshold=FINAL_THRESHOLD, margin_min=FINAL_MARGIN_MIN)
report = entity_level_report(preds, val_gt)
per_entity = report["per_entity"]
worst = sorted(per_entity.items(), key=lambda x: x[1][2])[:10]  # lowest F0.5 first

print("Worst-scoring entities:")
for s1_id, (p, r, f) in worst:
    true_matches = val_gt.get(s1_id, set())
    predicted = preds.get(s1_id, set())
    s1_name = records.loc[s1_id, "business_name"] if s1_id in records.index else "?"
    print(f"\n{s1_id} ({s1_name})  P={p:.2f} R={r:.2f} F0.5={f:.2f}")
    print(f"  Ground truth: {true_matches}")
    print(f"  Predicted:    {predicted}")
    for cid in (true_matches | predicted):
        tag = "TP" if cid in true_matches and cid in predicted else \
              "FN (missed)" if cid in true_matches else "FP (wrong)"
        name = records.loc[cid, "business_name"] if cid in records.index else "?"
        print(f"    [{tag}] {cid}: {name!r}")


## Cell 8 — save the model (only after you're happy with the score)

In [ ]:
MODEL_PATH = "model.json"
model.save_model(MODEL_PATH)
with open(MODEL_PATH + ".meta.json", "w") as f:
    json.dump(
        {"threshold": FINAL_THRESHOLD, "margin_min": FINAL_MARGIN_MIN, "features": ALL_FEATURE_COLUMNS},
        f, indent=2,
    )
print("Saved:", MODEL_PATH, " threshold:", FINAL_THRESHOLD, " margin_min:", FINAL_MARGIN_MIN)


## Cell 9 — inference on the real test set
Rebuilds candidates + features on `test_source1/2/3.tsv`, writes `candidate_pairs.tsv` **before** any model filtering (required by the spec), then `matching_results.tsv` after thresholding. Mirrors `predict.py` so you can also run this from the command line.

In [ ]:
%%time
test_source1 = pd.read_csv(f"{TEST_DIR}/test_source1.tsv", sep="\t", dtype=str)
test_source2 = pd.read_csv(f"{TEST_DIR}/test_source2.tsv", sep="\t", dtype=str)
test_source3 = pd.read_csv(f"{TEST_DIR}/test_source3.tsv", sep="\t", dtype=str)

test_candidates = generate_candidates(test_source1, test_source2, test_source3, top_k=TOP_K)

# candidate_pairs.tsv -- written BEFORE any model filtering, exactly what's fed to the matcher
cand_out = (
    test_candidates.dropna(subset=["candidate_entity_id"])
    .groupby("source1_entity_id")["candidate_entity_id"]
    .apply(lambda s: ",".join(sorted(set(s))))
    .reindex(test_source1["entity_id"].rename("source1_entity_id"), fill_value="")
    .reset_index()
    .rename(columns={"candidate_entity_id": "candidate_entity_ids"})
)
cand_out.to_csv(f"{OUT_DIR}/candidate_pairs.tsv", sep="\t", index=False)

test_records = pd.concat([test_source1, test_source2, test_source3], ignore_index=True).set_index("entity_id")
test_feat = build_pair_features(test_candidates, test_records)
test_feat["score"] = model.predict_proba(test_feat[ALL_FEATURE_COLUMNS])[:, 1]

keep = test_feat[test_feat["score"] >= FINAL_THRESHOLD]
if FINAL_MARGIN_MIN > 0:
    keep = keep[keep["score_margin"] >= FINAL_MARGIN_MIN]

match_out = (
    keep.groupby("source1_entity_id")["candidate_entity_id"]
    .apply(lambda s: ",".join(sorted(set(s))))
    .reindex(test_source1["entity_id"].rename("source1_entity_id"), fill_value="")
    .reset_index()
    .rename(columns={"candidate_entity_id": "matched_entity_ids"})
)
match_out.to_csv(f"{OUT_DIR}/matching_results.tsv", sep="\t", index=False)

print(f"Wrote {OUT_DIR}/candidate_pairs.tsv and {OUT_DIR}/matching_results.tsv")
match_out.head()


## Cell 10 — validate before submitting
Runs `utils/validate_submission.py` if it's present in the student resource pack.

In [ ]:
import subprocess, os

validator = "utils/validate_submission.py"
if os.path.exists(validator):
    result = subprocess.run(
        ["python3", validator,
         "--matching", f"{OUT_DIR}/matching_results.tsv",
         "--candidate", f"{OUT_DIR}/candidate_pairs.tsv",
         "--test-dir", TEST_DIR],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print(f"{validator} not found next to this notebook -- run it manually before submitting.")
